## Teste de inferência - Modelo XGboost

Este notebook tem como objetivo validar o comportamento do modelo treinado em cenários simulados de uso da API, utilizando inputs semelhantes aos que serão enviados pelo backend. Nenhum re-treino é realizado aqui. Apenas teste do modelo já pronto.

## 1. Carregando o modelo salvo

In [ ]:
!pip install -r /content/drive/MyDrive/dados_vra/requirements.txt

In [4]:
import joblib

modelo = joblib.load("/content/drive/MyDrive/dados_vra/models/modelo_atraso_voo_clima_xgb_te_v2.pkl")


## 2. Criando inputs de teste
⚠️ **Importante:**
O modelo ML espera as mesmas features usadas no treino(é diferente da API JSON que já vai ser processada pelo Backend com as informações que estão no contrato de integração)

### Input “Atrasado”

In [6]:
import pandas as pd

In [16]:
# Input de teste
input_atrasado = pd.DataFrame([{
    "icao_empresa_aerea": "TAM",
    "icao_aerodromo_origem": "SBGR",
    "icao_aerodromo_destino": "SBRJ",
    "faixa_horaria": "tarde",
    "hora_prevista_frac": 16.5,
    "tempo_voo_estimado": 1.2,
    "voos_no_slot": 42,
    "mes": 12,
    "eh_fim_de_semana": 1,
    "distancia_km": 350,
    "temp": 22.0,
    "prcp": 12.5,
    "wspd": 28.0,
    "prcp_bin": 1,
    "vento_forte": 1
}])

# Inferência
proba_atraso = modelo["pipeline"].predict_proba(input_atrasado)[0, 1]
pred_atraso = int(proba_atraso >= modelo["threshold_recomendado"])

# Status
status = "Atrasado" if pred_atraso == 1 else "Pontual"
print(f"Resultado final: {status}")

print("Previsão:", pred_atraso)
print("Probabilidade:", proba_atraso)


Resultado final: Atrasado
Previsão: 1
Probabilidade: 0.6386799


### Input "Pontual"

In [15]:
# Input de exemplo com BAIXA probabilidade de atraso
input_pontual = pd.DataFrame([{
    # Variáveis categóricas
    "icao_empresa_aerea": "AZU",
    "icao_aerodromo_origem": "SBCT",
    "icao_aerodromo_destino": "SBFI",
    "faixa_horaria": "madrugada",

    # Variáveis numéricas / operacionais
    "hora_prevista_frac": 5.0,          # 05:00
    "tempo_voo_estimado": 1.1,          # horas
    "voos_no_slot": 3,                  # slot tranquilo
    "mes": 4,                           # abril
    "eh_fim_de_semana": 0,              # dia útil

    # Distância
    "distancia_km": 530,

    # Clima
    "temp": 18.0,                       # temperatura amena
    "prcp": 0.0,                        # sem chuva
    "wspd": 6.0,                        # vento fraco
    "prcp_bin": 0,                      # sem chuva
    "vento_forte": 0                   # vento normal
}])

# Inferência
proba_pontual = modelo["pipeline"].predict_proba(input_pontual)[0, 1]
pred_pontual = int(proba_pontual >= modelo["threshold_recomendado"])

# Status
status = "Atrasado" if pred_pontual == 1 else "Pontual"
print(f"Resultado final: {status}")

print("Previsão:",pred_pontual)
print("Probabilidade:",proba_pontual)

Resultado final: Pontual
Previsão: 0
Probabilidade: 0.20588872
